# Demo A — Da PCAP a Grafo
**Master Cybersecurity LUISS — 29 maggio 2026**  
Antonio Formato

**Pipeline:** `tshark` → `pandas` → `networkx` → `pyvis`

**Tre viste progressive sullo stesso pcap:**
1. **Naive** — tutti i flussi, nessuna aggregazione (caos visivo voluto)
2. **Aggregata pesata** — gli hub emergono
3. **Filtrata (external destinations)** — potenziali C2/exfiltration

**Bonus didattico:** applichiamo gli stessi algoritmi visti nella teoria T2 (PageRank, connected components) al grafo costruito dal pcap. Chiusura del cerchio.

## 0. Setup
Installazione una tantum (se non già fatto):  
```bash
pip install pandas networkx pyvis
```
**Inoltre serve `tshark` nel PATH.** Su Windows: lo installi insieme a Wireshark, poi aggiungi `C:\Program Files\Wireshark` al PATH. Su macOS: `brew install wireshark`. Su Ubuntu: `apt install tshark`.

**Se non puoi/vuoi usare tshark**, salta direttamente al fallback (cella più sotto) che carica un CSV pre-estratto.

In [42]:
import subprocess
import ipaddress
import json
from pathlib import Path

import pandas as pd
import networkx as nx
from pyvis.network import Network

In [43]:
# ───── CONFIGURAZIONE ─────
PCAP = "pcaps/2026-05-11-macOS-malware-infection-traffic.pcap"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
FLOWS_CSV = OUTPUT_DIR / "flows.csv"

## 1. Estrazione flussi con tshark
Estraiamo IP sorgente, IP destinazione e porte. Filtriamo solo i pacchetti con IP layer.

In [44]:
TSHARK = r"C:\Program Files\Wireshark\tshark.exe"

def extract_flows(pcap_path, csv_path):
    """Estrae src/dst/porte da un pcap usando tshark."""
    cmd = [
        TSHARK, "-r", str(pcap_path),
        "-T", "fields",
        "-e", "ip.src",
        "-e", "ip.dst",
        "-e", "tcp.dstport",
        "-e", "udp.dstport",
        "-E", "separator=,",
        "-E", "header=n",
        "-E", "occurrence=f",  # un solo valore per campo (evita multi-IP su frame riassemblati)
        "-Y", "ip",   # solo pacchetti con IP
    ]
    with open(csv_path, "w") as f:
        subprocess.run(cmd, stdout=f, check=True)
    print(f"✓ Estratto in {csv_path}")

extract_flows(PCAP, FLOWS_CSV)

✓ Estratto in output\flows.csv


### Fallback: niente tshark
Se tshark non è disponibile, puoi pre-estrarre i flussi da linea di comando e mettere il CSV in `output/flows.csv`:
```bash
tshark -r sample.pcap -T fields -e ip.src -e ip.dst -e tcp.dstport -e udp.dstport -E separator=, -Y ip > output/flows.csv
```
Oppure: puoi anche generare flussi sintetici per provare il resto del notebook (cella commentata sotto).

In [14]:
# # Dati sintetici di prova — decommentare per testare senza pcap
# import random
# random.seed(42)
# internal_hosts = [f"10.0.0.{i}" for i in range(1, 8)]
# external_hosts = [f"203.0.113.{i}" for i in [10, 25, 42, 77]] + ["8.8.8.8"]
# c2 = "203.0.113.42"  # nostro "C2" finto: beaconing
# rows = []
# for _ in range(500):
#     src = random.choice(internal_hosts)
#     dst = random.choice(external_hosts + internal_hosts)
#     dport = random.choice([80, 443, 53, 22, 3389])
#     rows.append((src, dst, dport, ""))
# # beaconing fitto verso il C2
# for _ in range(200):
#     rows.append((random.choice(internal_hosts), c2, 443, ""))
# pd.DataFrame(rows, columns=["src","dst","tcp_dport","udp_dport"]).to_csv(FLOWS_CSV, index=False, header=False)
# print("✓ CSV sintetico creato")

## 2. Caricamento e ispezione

In [45]:
df = pd.read_csv(FLOWS_CSV, names=["src", "dst", "tcp_dport", "udp_dport"], dtype=str,
              on_bad_lines="skip")
df["dport"] = df["tcp_dport"].fillna(df["udp_dport"])
df = df.dropna(subset=["src", "dst"])

print(f"Flussi totali:      {len(df):,}")
print(f"IP src unici:       {df['src'].nunique()}")
print(f"IP dst unici:       {df['dst'].nunique()}")
print(f"Porte dst uniche:   {df['dport'].nunique()}")
df.head()

Flussi totali:      20,555
IP src unici:       38
IP dst unici:       42
Porte dst uniche:   181


,src,dst,tcp_dport,udp_dport,dport
0,10.5.11.101,10.5.11.1,NaN,53,53
1,10.5.11.101,10.5.11.1,NaN,53,53
2,10.5.11.1,10.5.11.101,NaN,62429,62429
3,10.5.11.1,10.5.11.101,NaN,50153,50153
4,10.5.11.101,142.250.114.113,443,NaN,443


## 3. Vista 1 — Grafo ingenuo (pyvis / vis.js)

Tutti i flussi, senza aggregazione. **Risultato: caos visivo.**  
*Pedagogicamente serve a mostrare perché aggregare è necessario.*

**Scelta intenzionale:** questa vista usa pyvis (libreria force-directed basata su vis.js) *senza tuning*. La fisica casuale produce un layout illeggibile — ed è esattamente il punto: dimostrare che un grafo denso senza filtraggio né layout semantico è inutile per un analista.

**Principio didattico:** le viste successive (2, 3, PageRank) usano Cytoscape.js con layout deterministici e semantici. Il contrasto tra questa vista e le successive mostra il valore delle scelte di rappresentazione.

In [46]:
G1 = nx.from_pandas_edgelist(df, "src", "dst", create_using=nx.DiGraph())
print(f"Nodi: {G1.number_of_nodes()}, Archi: {G1.number_of_edges()}")

net1 = Network(notebook=False, directed=True, height="650px", width="100%",
               bgcolor="#1a1a1a", font_color="white")
net1.from_nx(G1)
net1.write_html(str(OUTPUT_DIR / "view1_naive.html"), open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view1_naive.html'}")

Nodi: 42, Archi: 78
→ output\view1_naive.html


## 4. Vista 2 — Grafo aggregato e pesato

Stessi nodi. **L'arco src→dst ha peso = numero di flussi.** Gli hub emergono.

**Principi di visualizzazione:**
- **Layout concentrico** (Cytoscape.js): i nodi con traffico maggiore occupano il centro, quelli periferici gli anelli esterni. Questo rende immediato individuare gli *hub*.
- **Dimensione nodo ∝ √volume**: la radice quadrata evita che un singolo nodo massivo renda invisibili gli altri.
- **Colore condizionale**: i nodi con volume superiore alla soglia (top-3) si evidenziano in arancione — segnale visivo per l'analista.
- **Spessore arco ∝ peso normalizzato**: trasforma un numero (count) in un canale visivo pre-attentivo (larghezza della linea).

In [54]:
weighted = df.groupby(["src", "dst"]).size().reset_index(name="weight")
G2 = nx.from_pandas_edgelist(weighted, "src", "dst",
                              edge_attr="weight", create_using=nx.DiGraph())

# -- Metriche per dimensionamento nodi --
out_volume = weighted.groupby("src")["weight"].sum().to_dict()
in_volume = weighted.groupby("dst")["weight"].sum().to_dict()
max_weight = weighted["weight"].max()

# -- Cytoscape.js elements --
cy2_elements = []
for n in G2.nodes():
    total = out_volume.get(n, 0) + in_volume.get(n, 0)
    cy2_elements.append({"data": {"id": n, "label": n, "volume": int(total),
                                   "size": int(20 + (total ** 0.5) * 2)}})
for u, v, d in G2.edges(data=True):
    w = d.get("weight", 1)
    cy2_elements.append({"data": {"source": u, "target": v, "weight": int(w),
                                   "width": max(1, int(1 + (w / max_weight) * 8))}})

high_vol_threshold = sorted([out_volume.get(n, 0) + in_volume.get(n, 0) for n in G2.nodes()], reverse=True)[3]

# -- Tooltip CSS + JS snippet (riutilizzabile) --
TOOLTIP_CSS = (
    '#tooltip{position:fixed;background:#161b22;border:1px solid #30363d;color:#e6edf3;'
    'padding:10px 14px;border-radius:6px;font-size:12px;font-family:system-ui;'
    'pointer-events:none;display:none;white-space:pre-line;z-index:9999}'
)

def tooltip_js(content_expr):
    """Genera JS per mouseover/mousemove/mouseout tooltip."""
    return (
        'var tip=document.getElementById("tooltip");'
        'cy.on("mouseover","node",function(e){'
        'var d=e.target.data();'
        'tip.innerHTML=' + content_expr + ';'
        'tip.style.display="block";'
        '});'
        'cy.on("mousemove","node",function(e){'
        'tip.style.left=(e.originalEvent.clientX+12)+"px";'
        'tip.style.top=(e.originalEvent.clientY+12)+"px";'
        '});'
        'cy.on("mouseout","node",function(){tip.style.display="none";});'
    )

# -- Template Vista 2 --
v2_parts = [
    '<!DOCTYPE html><html><head><meta charset="utf-8">',
    '<title>Vista 2 - Grafo pesato</title>',
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/cytoscape/3.28.1/cytoscape.min.js"></script>',
    '<style>body{margin:0;background:#0d1117;overflow:hidden}',
    '#cy{width:100%;height:100vh}',
    TOOLTIP_CSS,
    '</style></head><body><div id="cy"></div><div id="tooltip"></div>',
    '<script>',
    'var cy=cytoscape({',
    'container:document.getElementById("cy"),',
    'elements:__ELEMENTS__,',
    'style:[',
    '{selector:"node",style:{"label":"data(label)","width":"data(size)","height":"data(size)",',
    '"background-color":"#6fb1fc","color":"#e6edf3","font-size":"11px",',
    '"text-valign":"bottom","text-margin-y":"5px","border-width":2,"border-color":"#2f81f7"}},',
    '{selector:"edge",style:{"width":"data(width)","line-color":"#3d5a80",',
    '"target-arrow-color":"#3d5a80","target-arrow-shape":"triangle","curve-style":"bezier","opacity":0.7}},',
    '{selector:"node[volume > __HIGH_VOL__]",style:{"background-color":"#f78166","border-color":"#da3633"}}',
    '],',
    'layout:{name:"concentric",concentric:function(n){return n.data("volume");},',
    'levelWidth:function(){return 2000;},minNodeSpacing:50,animate:false}',
    '});',
    tooltip_js('d.label+"<br>Volume: "+d.volume+" flussi"'),
    '</script></body></html>',
]
CYTO_TEMPLATE_V2 = ''.join(v2_parts)

html2 = CYTO_TEMPLATE_V2.replace('__ELEMENTS__', json.dumps(cy2_elements)).replace('__HIGH_VOL__', str(high_vol_threshold))
(OUTPUT_DIR / "view2_weighted.html").write_text(html2, encoding="utf-8")
print(f"-> {OUTPUT_DIR / 'view2_weighted.html'}  ({len(cy2_elements)} elements)")

-> output\view2_weighted.html  (120 elements)


## 5. Vista 3 — Solo destinazioni esterne

Filtriamo: **flussi da host interno (RFC1918) verso destinazioni pubbliche.**  
Quello che resta sono i potenziali target di C2 o exfiltration.

**Principi di visualizzazione:**
- **Layout gerarchico top-down (dagre)**: la direzione del flusso è codificata nella posizione verticale — *interni in alto, esterni in basso*. L'occhio segue naturalmente il flusso dati verso l'esterno.
- **Codice colore semantico**: blu = host RFC1918 (fidato), rosso = destinazione pubblica (da investigare). Questo mapping è consistente con le convenzioni SOC.
- **Evidenziazione automatica dei top-N**: i nodi esterni con più flussi inbound ricevono un bordo più spesso e un colore più intenso — sono i candidati C2/exfil senza bisogno di leggere numeri.
- **Legenda integrata**: in demo proiettata, la legenda on-screen evita di dover spiegare i colori a voce.

**Perché dagre?** In un grafo bipartito (interni → esterni) il layout gerarchico minimizza il crossing degli archi e rende la topologia leggibile anche con 40 nodi.

In [55]:
# Solo RFC1918 conta come "interno".
RFC1918 = [ipaddress.ip_network(n) for n in
           ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]

def is_internal(ip):
    try:
        addr = ipaddress.ip_address(ip)
        return any(addr in net for net in RFC1918)
    except ValueError:
        return False

ext = weighted[
    weighted["src"].apply(is_internal) &
    ~weighted["dst"].apply(is_internal)
]
print(f"Host interni che parlano verso l'esterno: {ext['src'].nunique()}")
print(f"Destinazioni esterne distinte:            {ext['dst'].nunique()}")

G3 = nx.from_pandas_edgelist(ext, "src", "dst",
                              edge_attr="weight", create_using=nx.DiGraph())

# -- Cytoscape.js: layout gerarchico top-down (interni sopra, esterni sotto) --
max_w3 = ext["weight"].max()
cy3_elements = []
for n in G3.nodes():
    internal = is_internal(n)
    w_total = int(ext.loc[ext["dst"] == n, "weight"].sum()) if not internal else 0
    cy3_elements.append({"data": {"id": n, "label": n,
                                   "type": "internal" if internal else "external",
                                   "size": 30 if internal else int(18 + (w_total ** 0.4) * 3),
                                   "weight_in": w_total}})
for u, v, d in G3.edges(data=True):
    w = d.get("weight", 1)
    cy3_elements.append({"data": {"source": u, "target": v, "weight": int(w),
                                   "width": max(1, int(1 + (w / max_w3) * 10))}})

c2_thresh = sorted([e["data"].get("weight_in", 0) for e in cy3_elements if e["data"].get("type") == "external"], reverse=True)[2]
edge_hi = sorted([d.get("weight", 1) for _, _, d in G3.edges(data=True)], reverse=True)[2]

# -- Template Vista 3 --
v3_parts = [
    '<!DOCTYPE html><html><head><meta charset="utf-8">',
    '<title>Vista 3 - Interno verso Esterno</title>',
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/cytoscape/3.28.1/cytoscape.min.js"></script>',
    '<script src="https://unpkg.com/dagre@0.8.5/dist/dagre.min.js"></script>',
    '<script src="https://unpkg.com/cytoscape-dagre@2.5.0/cytoscape-dagre.js"></script>',
    '<style>',
    'body{margin:0;background:#0d1117;overflow:hidden;font-family:system-ui}',
    '#cy{width:100%;height:100vh}',
    '#legend{position:fixed;top:16px;right:16px;background:#161b22;border:1px solid #30363d;',
    'border-radius:8px;padding:12px 16px;color:#e6edf3;font-size:13px}',
    '.dot{display:inline-block;width:12px;height:12px;border-radius:50%;margin-right:6px;vertical-align:middle}',
    TOOLTIP_CSS,
    '</style></head><body>',
    '<div id="cy"></div><div id="tooltip"></div>',
    '<div id="legend">',
    '<b>Legenda</b><br>',
    '<span class="dot" style="background:#58a6ff"></span> Host interno (RFC1918)<br>',
    '<span class="dot" style="background:#f85149"></span> Destinazione esterna<br>',
    '<span style="color:#8b949e">Spessore arco proporzionale ai flussi</span>',
    '</div>',
    '<script>',
    'var cy=cytoscape({',
    'container:document.getElementById("cy"),',
    'elements:__ELEMENTS__,',
    'style:[',
    '{selector:"node[type=\\"internal\\"]",style:{',
    '"label":"data(label)","width":"data(size)","height":"data(size)",',
    '"background-color":"#58a6ff","color":"#e6edf3","font-size":"13px",',
    '"font-weight":"bold","text-valign":"top","text-margin-y":"-8px",',
    '"border-width":3,"border-color":"#1f6feb"}},',
    '{selector:"node[type=\\"external\\"]",style:{',
    '"label":"data(label)","width":"data(size)","height":"data(size)",',
    '"background-color":"#f85149","color":"#e6edf3","font-size":"10px",',
    '"text-valign":"bottom","text-margin-y":"5px",',
    '"border-width":2,"border-color":"#da3633"}},',
    '{selector:"node[weight_in > __C2_THRESH__]",style:{',
    '"background-color":"#ff7b72","border-color":"#f85149","border-width":4,',
    '"font-size":"12px","font-weight":"bold"}},',
    '{selector:"edge",style:{',
    '"width":"data(width)","line-color":"#484f58","target-arrow-color":"#6e7681",',
    '"target-arrow-shape":"triangle","curve-style":"bezier","opacity":0.6}},',
    '{selector:"edge[weight > __EDGE_HI__]",style:{',
    '"line-color":"#f0883e","target-arrow-color":"#f0883e","opacity":0.9}}',
    '],',
    'layout:{name:"dagre",rankDir:"TB",nodeSep:60,rankSep:120,animate:false}',
    '});',
    tooltip_js('d.label+"<br>Tipo: "+d.type+"<br>Flussi inbound: "+d.weight_in'),
    '</script></body></html>',
]
CYTO_TEMPLATE_V3 = ''.join(v3_parts)

html3 = (CYTO_TEMPLATE_V3
         .replace('__ELEMENTS__', json.dumps(cy3_elements))
         .replace('__C2_THRESH__', str(c2_thresh))
         .replace('__EDGE_HI__', str(edge_hi)))
(OUTPUT_DIR / "view3_external.html").write_text(html3, encoding="utf-8")
print(f"-> {OUTPUT_DIR / 'view3_external.html'}  ({ext['dst'].nunique()} destinazioni esterne)")

Host interni che parlano verso l'esterno: 1
Destinazioni esterne distinte:            39
-> output\view3_external.html  (39 destinazioni esterne)


## 6. Bonus — Stessi algoritmi della teoria, applicati live

Nella parte teorica T2 abbiamo visto PageRank, community detection, connected components.  
**Ora li applichiamo al grafo costruito dal pcap.**  
È il momento in cui la teoria diventa visibile.

**Collegamento pedagogico:**
- **PageRank** (Brin & Page, 1998): originariamente pensato per classificare pagine web, qui classifica nodi di rete per *centralità strutturale*. Un IP C2 con comunicazione bidirezionale intensa verso la vittima accumula score.
- **Connected components**: identifica cluster isolati di comunicazione. In un pcap reale di un SOC con più subnet, componenti separate possono indicare segmentazione di rete o traffico laterale confinato.
- **Dalla teoria alla pratica**: lo studente vede che gli stessi 3 concetti (grafo, algoritmo, visualizzazione) si applicano identicamente a pacchetti IP, identità Active Directory (Demo B), o entità CTI (Demo D).

### 6.1 PageRank — quali IP sono "centrali" nel traffico?

In [50]:
pr = nx.pagerank(G2, weight="weight")
top10 = sorted(pr.items(), key=lambda x: -x[1])[:10]

print("Top 10 nodi per PageRank:\n")
print(f"{'IP':<20} {'Score':<10} {'Tipo'}")
print("-" * 45)
for ip, score in top10:
    tipo = "interno" if is_internal(ip) else "ESTERNO"
    print(f"{ip:<20} {score:<10.4f} {tipo}")

Top 10 nodi per PageRank:

IP                   Score      Tipo
---------------------------------------------
10.5.11.101          0.4542     interno
165.245.215.18       0.3182     ESTERNO
172.67.165.232       0.0135     ESTERNO
208.54.62.240        0.0119     ESTERNO
172.67.198.113       0.0111     ESTERNO
142.251.116.94       0.0110     ESTERNO
94.232.249.129       0.0085     ESTERNO
17.253.126.69        0.0072     ESTERNO
10.5.11.1            0.0071     interno
142.250.114.113      0.0071     ESTERNO


### 6.2 Connected components — cluster di comunicazione

In [51]:
G2_undirected = G2.to_undirected()
components = list(nx.connected_components(G2_undirected))
components.sort(key=len, reverse=True)

print(f"Numero di componenti connesse: {len(components)}\n")
for i, comp in enumerate(components[:5], 1):
    print(f"Componente #{i}: {len(comp)} nodi  — esempio: {list(comp)[:3]}")

Numero di componenti connesse: 1

Componente #1: 42 nodi  — esempio: ['165.245.215.18', '151.101.2.137', '208.54.62.240']


### 6.3 Visualizzazione finale con PageRank come colore

**Principi di visualizzazione:**
- **Layout concentrico per PageRank**: il nodo con score più alto sta al centro, i periferici si allontanano. L'importanza strutturale diventa posizione spaziale.
- **Doppia codifica (ridondanza)**: lo stesso dato (PageRank) è mappato sia su *dimensione* che su *colore* (gradiente blu→rosso). Questo rispetta il principio di ridondanza dei canali visivi: chi non distingue bene i colori legge comunque la dimensione.
- **Bar-chart overlay**: i top-5 sono mostrati anche in formato tabulare nell'angolo, per dare un ancoraggio numerico alla visualizzazione spaziale.
- **Click interattivo**: in demo, il presentatore può cliccare un nodo sospetto per mostrare il valore esatto di PageRank — passaggio dalla visione d'insieme al dettaglio.

**Perché PageRank funziona qui?** In un grafo di traffico di rete, PageRank non misura solo "chi riceve più pacchetti" (quello è in-degree pesato), ma *chi riceve traffico da nodi a loro volta importanti*. Un C2 che comunica bidirezionalmente con una vittima molto attiva eredita importanza dalla vittima stessa, amplificando il suo score rispetto a un semplice conteggio di flussi.

In [56]:
# -- Cytoscape.js: PageRank visualizzato come gradiente + dimensione --
pr_max = max(pr.values()) if pr else 1

def pr_color(score, mx):
    """Gradiente da blu (#58a6ff) a rosso (#f85149) proporzionale al PageRank."""
    t = score / mx
    r = int(88 + (248 - 88) * t)
    g = int(166 - (166 - 81) * t)
    b = int(255 - (255 - 73) * t)
    return f"rgb({r},{g},{b})"

cy_pr_elements = []
for n in G2.nodes():
    s = pr.get(n, 0)
    cy_pr_elements.append({"data": {
        "id": n, "label": n,
        "pr_score": round(s, 5),
        "pr_norm": round(s / pr_max, 3),
        "size": int(20 + (s / pr_max) * 80),
        "color": pr_color(s, pr_max),
        "type": "internal" if is_internal(n) else "external"
    }})
for u, v, d in G2.edges(data=True):
    w = d.get("weight", 1)
    cy_pr_elements.append({"data": {"source": u, "target": v, "weight": int(w),
                                     "width": max(1, int(1 + (w / max_weight) * 6))}})

# Mini bar-chart HTML per top-5
top5 = sorted(pr.items(), key=lambda x: -x[1])[:5]
bars_html = ""
for ip, s in top5:
    pct = int((s / pr_max) * 100)
    color = pr_color(s, pr_max)
    tipo_tag = " EXT" if not is_internal(ip) else ""
    bars_html += (
        '<div class="bar-row"><span class="bar-label">'
        + ip + tipo_tag
        + '</span><div class="bar" style="width:'
        + str(pct) + '%;background:' + color
        + '"></div><span class="bar-val">'
        + f"{s:.4f}" + '</span></div>'
    )

# -- Template PageRank --
pr_parts = [
    '<!DOCTYPE html><html><head><meta charset="utf-8">',
    '<title>PageRank - Chi e centrale nel traffico?</title>',
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/cytoscape/3.28.1/cytoscape.min.js"></script>',
    '<style>',
    'body{margin:0;background:#0d1117;overflow:hidden;font-family:system-ui}',
    '#cy{width:100%;height:100vh}',
    '#info{position:fixed;top:16px;left:16px;background:#161b22;border:1px solid #30363d;',
    'border-radius:8px;padding:14px 18px;color:#e6edf3;font-size:13px;max-width:320px}',
    '#info h3{margin:0 0 8px;color:#f0883e}',
    '#bar-chart{margin-top:10px}',
    '.bar-row{display:flex;align-items:center;margin:3px 0}',
    '.bar-label{width:130px;font-size:11px;color:#8b949e;text-align:right;padding-right:8px}',
    '.bar{height:14px;border-radius:3px;min-width:2px}',
    '.bar-val{font-size:10px;color:#8b949e;padding-left:4px}',
    TOOLTIP_CSS,
    '</style></head><body>',
    '<div id="cy"></div><div id="tooltip"></div>',
    '<div id="info">',
    '<h3>PageRank Top 5</h3>',
    '<div id="bar-chart">__BARS__</div>',
    '<p style="color:#6e7681;font-size:11px;margin-top:12px">',
    'Dimensione nodo proporzionale al PageRank<br>',
    'Colore: blu (basso) - rosso (alto)<br>',
    'Hover su un nodo per dettagli',
    '</p></div>',
    '<script>',
    'var cy=cytoscape({',
    'container:document.getElementById("cy"),',
    'elements:__ELEMENTS__,',
    'style:[',
    '{selector:"node",style:{',
    '"label":"data(label)","width":"data(size)","height":"data(size)",',
    '"background-color":"data(color)","color":"#e6edf3",',
    '"font-size":"10px","text-valign":"bottom","text-margin-y":"5px",',
    '"border-width":2,"border-color":"#30363d"}},',
    '{selector:"node[pr_norm > 0.5]",style:{',
    '"font-size":"13px","font-weight":"bold","border-width":3,',
    '"text-valign":"top","text-margin-y":"-8px"}},',
    '{selector:"edge",style:{',
    '"width":"data(width)","line-color":"#21262d","target-arrow-color":"#484f58",',
    '"target-arrow-shape":"triangle","curve-style":"bezier","opacity":0.4}}',
    '],',
    'layout:{name:"concentric",',
    'concentric:function(n){return n.data("pr_norm")*100;},',
    'levelWidth:function(){return 8;},',
    'minNodeSpacing:40,animate:false}',
    '});',
    tooltip_js('d.label+"<br>PageRank: "+d.pr_score+"<br>Tipo: "+d.type'),
    '</script></body></html>',
]
CYTO_TEMPLATE_PR = ''.join(pr_parts)

html_pr = (CYTO_TEMPLATE_PR
           .replace('__ELEMENTS__', json.dumps(cy_pr_elements))
           .replace('__BARS__', bars_html))
(OUTPUT_DIR / "view_bonus_pagerank.html").write_text(html_pr, encoding="utf-8")
print(f"-> {OUTPUT_DIR / 'view_bonus_pagerank.html'}  (top: {top5[0][0]} = {top5[0][1]:.4f})")

-> output\view_bonus_pagerank.html  (top: 10.5.11.101 = 0.4542)


## 7. Recap — cosa hai mostrato in aula

1. **Da un pcap di un singolo cliente/lab → un grafo interrogabile** in <30 righe di Python  
2. **Tre rappresentazioni progressive** dello stesso oggetto matematico (lo stesso pcap)  
3. **Gli algoritmi astratti della teoria** (PageRank, connected components) **diventano concreti** sul traffico reale  
4. **Bridge naturale verso BloodHound** (demo successiva): stesso paradigma, dominio diverso (identità AD invece di pacchetti IP)